# Exercise 01 : Graphs and Loops

- A graph is nodes over typed state, joined by edges
- A conditional edge reads the state and names the next node
- `Command` updates the state and chooses the next node in one return

You will build a linear graph, add a router, add a loop with a limit, and
route with `Command`. The Aurelia records are made up.

Worked examples to revisit : module 02, demo02 · nodes-and-edges, demo03 · routing,
demo04 · command-and-send, demo06 · a-model-in-a-node.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [ ]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

## Step 1 : Linear graph

A ticket from T-1001 passes through `normalise` and then `stamp`.

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class Ticket(TypedDict):
    text: str
    priority: str
    route: str


def normalise(state: Ticket) -> dict:
    return {"text": " ".join(state["text"].split()).strip()}


def stamp(state: Ticket) -> dict:
    return {"text": "[TICKET] " + state["text"]}


builder = StateGraph(Ticket)
builder.add_node("normalise", normalise)
builder.add_node(____, stamp)
builder.add_edge(____, "normalise")
builder.add_edge("normalise", ____)
builder.add_edge("stamp", ____)
graph = builder.compile()
linear = graph.invoke({"text": "  Slip ring   communication loss reported by radiology staff  ",
                       "priority": "", "route": ""})
print(linear)

<details><summary>Hint</summary>

`"stamp"`, `START`, `"stamp"` and `END`.

</details>

In [ ]:
expected = "[TICKET] Slip ring communication loss reported by radiology staff"
print("check :", "pass" if linear["text"] == expected else "fail")
print("text :", linear["text"])

## Step 2 : Router

A model node sets the priority to `high` or `low`. A router sends a high
ticket to `page_engineer` and a low ticket to `queue`. The second ticket is made up.

In [ ]:
def prioritise(state: Ticket) -> dict:
    reply = llm.invoke("Classify the ticket. Reply high when scanning or other work is stopped now, "
                       "low for anything else. Reply with one word.\n\n"
                       + state["text"])
    print("usage :", reply.usage_metadata)
    label = reply.content.strip().lower()
    return {"priority": "high" if label.startswith("high") else "low"}


def route(state: Ticket) -> Literal["page_engineer", "queue"]:
    return ____


router = StateGraph(Ticket)
router.add_node("prioritise", prioritise)
router.add_node("page_engineer", lambda s: {"route": "engineer paged"})
router.add_node("queue", lambda s: {"route": "added to the queue"})
router.add_edge(START, "prioritise")
router.____("prioritise", route)
router.add_edge("page_engineer", END)
router.add_edge("queue", END)
router_graph = router.compile()
runs = []
for text in ["Gantry over-temperature during the afternoon list, scanning is stopped.",
             "Please file the CT-400 fan procedure in the site folder."]:
    item = router_graph.invoke({"text": text, "priority": "", "route": ""})
    runs.append(item)
    print(item["priority"], "->", item["route"])

<details><summary>Hint</summary>

`"page_engineer" if state["priority"] == "high" else "queue"` and `add_conditional_edges`.

</details>

In [ ]:
ok = len(runs) == 2 and all(
    item["route"] == ("engineer paged" if item["priority"] == "high" else "added to the queue")
    for item in runs)
print("check :", "pass" if ok else "fail")

## Step 3 : Loop

`shorten` rewrites a ticket until it is at most 60 characters, or three
attempts have been made.

In [ ]:
class Short(TypedDict):
    text: str
    attempts: int


def shorten(state: Short) -> dict:
    reply = llm.invoke(f"Rewrite in at most 60 characters, keep the meaning, no quotes : {state['text']}")
    print("usage :", reply.usage_metadata)
    text = reply.content.strip()
    print(f"attempt {state['attempts'] + 1} : {len(text)} chars | {text}")
    return {"text": text, "attempts": state["attempts"] + 1}


def again(state: Short) -> Literal["shorten", "__end__"]:
    if len(state["text"]) <= ____ or state["attempts"] >= ____:
        return END
    return ____


loop = StateGraph(Short)
loop.add_node("shorten", shorten)
loop.add_edge(START, "shorten")
loop.add_conditional_edges("shorten", again)
result = loop.compile().invoke({
    "text": "The CT-400 gantry at Kanaalzicht Medical Centre reports over-temperature during the afternoon "
            "list and scanning is stopped until an engineer arrives.",
    "attempts": 0})

<details><summary>Hint</summary>

`60`, `3` and `"shorten"`.

</details>

In [ ]:
stopped = result["attempts"] <= 3 and (len(result["text"]) <= 60 or result["attempts"] == 3)
print("attempts :", result["attempts"], "chars :", len(result["text"]))
print("check :", "pass" if stopped else "fail")

## Step 4 : Command

`triage` already knows the priority. Return a `Command` that stores the
chosen node on `assigned` and goes there. The destination node writes `route`.

In [ ]:
from langgraph.types import Command


class Visit(TypedDict):
    priority: str
    assigned: str
    route: str


def triage(state: Visit) -> Command[Literal["page", "queue"]]:
    target = "page" if state["priority"] == "high" else "queue"
    return ____


routed = StateGraph(Visit)
routed.add_node("triage", triage)
routed.add_node("page", lambda s: {"route": "engineer paged"})
routed.add_node("queue", lambda s: {"route": "added to the queue"})
routed.add_edge(START, "triage")
routed.add_edge("page", END)
routed.add_edge("queue", END)
command_graph = routed.compile()
high = command_graph.invoke({"priority": "high", "assigned": "", "route": ""})
low = command_graph.invoke({"priority": "low", "assigned": "", "route": ""})
print("high :", high)
print("low :", low)

<details><summary>Hint</summary>

`Command(update={"assigned": target}, goto=target)`.

</details>

In [ ]:
ok = (high["assigned"] == "page" and high["route"] == "engineer paged"
      and low["assigned"] == "queue" and low["route"] == "added to the queue")
print("check :", "pass" if ok else "fail")

## If time permits

- Print `router_graph.get_graph().draw_mermaid()` and compare it with the router slide
- Give the loop in Step 3 a `recursion_limit` of 2 and read the error
- Fan out with `Send`, one branch per site in `sites = ["S01", "S02", "S03", "S04"]`, and
  collect the site ids on a key typed `Annotated[list[str], operator.add]`